# 2D brightfield segmentation with nnU-Net v2

- Two independent 2D nnU-Net models trained from single-channel brightfield `.tiff` images with matching (binarised) label masks: one for **organoids**, one for **rosettes**.
- Each image/label pair is converted to NIfTI before training. Pixel spacing is read from each image's TIFF header; labels have no spacing metadata of their own, so they reuse the spacing of their paired image.
- Images (for both training and prediction) may have varying pixel sizes -> nnU-Net resamples to a consistent physical spacing during preprocessing/inference, provided the true spacing is embedded in every case.
- Note! This is a semantic segmentation -> touching objects are merged and may require watershed splitting.

In [17]:
import os, json, csv, subprocess
from pathlib import Path
import numpy as np
import tifffile
import SimpleITK as sitk
import torch
import matplotlib.pyplot as plt
from nnunetv2.dataset_conversion.generate_dataset_json import generate_dataset_json

print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

torch 2.12.0.dev20260217+cu128 | cuda True | NVIDIA GeForce RTX 5080


In [18]:
def tif_pixel_sizes(tif):
    """Return (x_um, y_um) from a tifffile TiffFile; either value is None if unavailable."""
    x_um = y_um = None
    tags = {t.name: t.value for t in tif.pages[0].tags.values()}
    xr = tags.get("XResolution")
    yr = tags.get("YResolution")
    if xr and xr[0]:
        x_um = xr[1] / xr[0]  # XResolution = pixels per unit -> unit per pixel
    if yr and yr[0]:
        y_um = yr[1] / yr[0]
    return x_um, y_um


def get_spacing_xy(img_path):
    """Per-image (x, y) pixel spacing read from the TIFF header.
    Labels carry no resolution tags of their own -> callers should reuse the spacing of their paired image.
    Raises if the tags are missing -> run check_pixel_metadata() first to catch this up front."""
    with tifffile.TiffFile(str(img_path)) as tif:
        x_um, y_um = tif_pixel_sizes(tif)
    if x_um is None or y_um is None:
        raise ValueError(f'{Path(img_path).name}: missing XResolution/YResolution pixel-size tag')
    return (x_um, y_um)


def check_pixel_metadata(img_paths):
    """Flag any images missing an XResolution/YResolution pixel-size tag, so they can be fixed or
    removed before the main conversion/prediction pipeline runs. Returns the list of offending paths."""
    missing = []
    for p in img_paths:
        with tifffile.TiffFile(str(p)) as tif:
            x_um, y_um = tif_pixel_sizes(tif)
        if x_um is None or y_um is None:
            missing.append(p)
    if missing:
        print(f'! {len(missing)}/{len(img_paths)} image(s) missing pixel-size metadata:')
        for p in missing:
            print('   ', p)
    else:
        print(f'all {len(img_paths)} images have pixel-size metadata')
    return missing

In [19]:
def load_pair(img_path, labels_dir, label_ext):
    """Brightfield (Y,X float32) + binarised foreground mask (Y,X uint8) + (x,y) pixel spacing (um) from a matched pair.
    The label shares the image stem but lives in labels_dir with extension label_ext and has no pixel-size metadata
    of its own -> the image's spacing is reused for it."""
    bf = np.asarray(tifffile.imread(str(img_path))).astype(np.float32)
    lab_path = labels_dir / (img_path.stem + label_ext)
    inst = np.asarray(tifffile.imread(str(lab_path)))
    if bf.shape != inst.shape:
        raise ValueError(f'{img_path.name}: image {bf.shape} vs label {inst.shape} shape mismatch')
    mask = (inst > 0).astype(np.uint8)          # instance ids -> single foreground class
    spacing_xy = get_spacing_xy(img_path)
    return bf, mask, spacing_xy


def write_nii_2d(arr_yx, path, is_label, spacing_xy):
    """Write a 2D array as a (1,Y,X) NIfTI; the singleton z makes nnU-Net treat it as 2D."""
    vol = arr_yx[None]
    img = sitk.GetImageFromArray(vol.astype(np.uint8 if is_label else np.float32))
    img.SetSpacing((float(spacing_xy[0]), float(spacing_xy[1]), 999.0))   # sitk order (x,y,z)
    sitk.WriteImage(img, str(path))

In [34]:
def build_dataset(img_paths, labels_dir, label_ext, raw_ds):
    """Convert every matched (image, label) pair into nnU-Net raw imagesTr/labelsTr NIfTIs.
    Writes a provenance.csv alongside raw_ds and returns the per-case rows."""
    (raw_ds / 'imagesTr').mkdir(parents=True, exist_ok=True)
    (raw_ds / 'labelsTr').mkdir(parents=True, exist_ok=True)
    rows = []
    spacings = []
    for k, ip in enumerate(img_paths):
        case = f'case_{k:03d}'
        bf, mask, spacing_xy = load_pair(ip, labels_dir, label_ext)
        write_nii_2d(bf,   raw_ds / 'imagesTr' / f'{case}_0000.nii.gz', is_label=False, spacing_xy=spacing_xy)
        write_nii_2d(mask, raw_ds / 'labelsTr' / f'{case}.nii.gz',      is_label=True,  spacing_xy=spacing_xy)
        frac = 100.0 * float(mask.mean())
        spacings.append(spacing_xy)
        rows.append((case, ip.stem, round(frac, 2), round(spacing_xy[0], 4), round(spacing_xy[1], 4)))
        print(f'  {case}  {ip.stem[:44]:44s}  foreground={frac:5.2f}%  spacing_xy={spacing_xy}')
    with open(raw_ds / 'provenance.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['case', 'source_stem', 'foreground_pct', 'spacing_x_um', 'spacing_y_um'])
        w.writerows(rows)
    xs = sorted(s[0] for s in spacings)
    if xs and (xs[-1] - xs[0]) > 1e-6:
        print(f'  ! spacing varies across this dataset: x_um {xs[0]:.4f} - {xs[-1]:.4f} '
              '(fine for nnU-Net, which resamples to a target spacing during preprocessing)')
    return rows

In [35]:
def setup_nnunet_dirs(nnunet_base):
    """Create/point nnU-Net at raw/preprocessed/results folders under nnunet_base and set its env vars.
    Returns a dict with the three paths."""
    nnunet_raw          = nnunet_base / 'nnUNet_raw'
    nnunet_preprocessed = nnunet_base / 'nnUNet_preprocessed'
    nnunet_results      = nnunet_base / 'nnUNet_results'
    for d in (nnunet_raw, nnunet_preprocessed, nnunet_results):
        d.mkdir(parents=True, exist_ok=True)
    os.environ['nnUNet_raw']          = str(nnunet_raw)
    os.environ['nnUNet_preprocessed'] = str(nnunet_preprocessed)
    os.environ['nnUNet_results']      = str(nnunet_results)
    os.environ.pop('nnUNet_n_proc_DA', None)   # clear stale value
    return {'raw': nnunet_raw, 'preprocessed': nnunet_preprocessed, 'results': nnunet_results}


def run(cmd, da_workers=None):
    """Stream an nnU-Net CLI command. da_workers=0 -> single-threaded augmentation (Windows)."""
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    if da_workers is not None:
        env['nnUNet_n_proc_DA'] = str(da_workers)
        env.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
        env['OMP_NUM_THREADS'] = '1'
        env['MKL_NUM_THREADS'] = '1'
    print('>', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, env=env, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

In [36]:
def read_stack_bf(path):
    """(Z, Y, X) float32 brightfield from a stacked TIFF; channel 0 if a channel axis exists."""
    arr = np.asarray(tifffile.imread(str(path))).astype(np.float32)
    if arr.ndim == 2:
        arr = arr[None]
    elif arr.ndim == 4:                                   # (Z, C, Y, X) -> channel 0
        arr = np.take(arr, 0, axis=1 if arr.shape[1] <= 4 else 0)
    return arr


def stage_stack(stack_path, stage_dir, index):
    """Write every slice of one time-lapse stack as an independent 2D NIfTI case.
    Returns (spacing_xy, n_slices)."""
    bf = read_stack_bf(stack_path)
    spacing_xy = get_spacing_xy(stack_path)   # one spacing per stack, shared by all its slices
    for z in range(bf.shape[0]):
        write_nii_2d(bf[z], stage_dir / f'{index:03d}_{z:04d}_0000.nii.gz', is_label=False, spacing_xy=spacing_xy)
    return spacing_xy, bf.shape[0]


def reassemble_stack(stack_path, pred_dir, index, n_slices, foreground_label_value, stack_src_root, stack_out_root):
    """Rebuild ZCYX (ch0=brightfield, ch1=foreground label) for one stack from its per-slice predictions.
    Returns (output_path, labels_array)."""
    bf = read_stack_bf(stack_path)
    labels = np.zeros(bf.shape, dtype=np.uint16)
    for z in range(n_slices):
        pred = sitk.GetArrayFromImage(sitk.ReadImage(str(pred_dir / f'{index:03d}_{z:04d}.nii.gz')))
        labels[z] = (pred[0] == foreground_label_value).astype(np.uint16)
    stack = np.stack([bf.astype(np.uint16), labels], axis=1)
    rel = stack_path.relative_to(stack_src_root)
    out = stack_out_root / rel.with_name(rel.stem + '_labels.tif')
    out.parent.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(str(out), stack, imagej=True, metadata={'axes': 'ZCYX'})
    return out, labels

In [23]:
def segment_folder(tifs, pred_in, pred_out, pred_tif_dir, dataset_id, config, fold, trainer, checkpoint,
                    foreground_label_value):
    """Predict a folder of new brightfield *.tiff frames and write a binary foreground mask *.tif per image.
    Raises if any image is missing pixel-size metadata -> fix/remove it and rerun."""
    tifs = list(tifs)
    bad_spacing = check_pixel_metadata(tifs)
    if bad_spacing:
        raise ValueError(f'{len(bad_spacing)} image(s) missing pixel-size metadata -> remove or fix them, then rerun')
    for d in (pred_in, pred_out, pred_tif_dir):
        d.mkdir(parents=True, exist_ok=True)
    meta = {}
    for k, tif in enumerate(tifs):
        bf = np.asarray(tifffile.imread(str(tif))).astype(np.float32)
        case = f'pred_{k:03d}'
        spacing_xy = get_spacing_xy(tif)
        write_nii_2d(bf, pred_in / f'{case}_0000.nii.gz', is_label=False, spacing_xy=spacing_xy)
        meta[case] = tif
    run(f'nnUNetv2_predict -i {pred_in} -o {pred_out} -d {dataset_id} -c {config} '
        f'-f {fold} -tr {trainer} -chk {checkpoint}')
    for case, tif in meta.items():
        pred = sitk.GetArrayFromImage(sitk.ReadImage(str(pred_out / f'{case}.nii.gz')))
        mask = (pred[0] == foreground_label_value).astype(np.uint8)   # drop singleton z
        out = pred_tif_dir / f'{tif.stem}_mask.tif'
        tifffile.imwrite(str(out), mask)
        print('wrote', out.name, '| foreground%=', round(100 * float(mask.mean()), 2))
    print('masks ->', pred_tif_dir)

## Configure the two tasks (organoid & rosette)

In [41]:
nnunet_base = Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\nnUNet")  # alongside the image/label pairs

tasks = [
    {
        'name': 'organoid',
        'dataset_id': 1,
        'dataset_name': 'Dataset001_Organoid',
        'images_dir': Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\organoid\tiff"), 
        'labels_dir': Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\organoid\labels"),            
        'image_glob': '*.tiff',
        'label_ext': '.tif',                        # labels share the image stem but use .tif
        'channel_names': {0: 'brightfield'},
        'labels': {'background': 0, 'organoid': 1},
        'foreground_label_value': 1,
        'config': '2d',
        'fold': 'all',                              # train on all cases; use a 0-4 fold for a CV Dice curve
        'trainer': 'nnUNetTrainer_250epochs_preview',  # 250-epoch trainer that also saves preview PNGs
        'checkpoint': 'checkpoint_best.pth',
    },
    {
        'name': 'rosette',
        'dataset_id': 2,
        'dataset_name': 'Dataset002_Rosette',
        'images_dir': Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\tiff\completed"),
        'labels_dir': Path(r"Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\labels\completed"),
        'image_glob': '*.tiff',
        'label_ext': '.tif',
        'channel_names': {0: 'brightfield'},
        'labels': {'background': 0, 'rosette': 1},
        'foreground_label_value': 1,
        'config': '2d',
        'fold': 'all',
        'trainer': 'nnUNetTrainer_250epochs_preview',
        'checkpoint': 'checkpoint_best.pth',
    },
]

nnunet_dirs = setup_nnunet_dirs(nnunet_base)
for task in tasks:
    task['raw_ds'] = nnunet_dirs['raw'] / task['dataset_name']
    print(f"{task['name']:8s} images: {task['images_dir']}")
    print(f"{task['name']:8s} labels: {task['labels_dir']}")
    print(f"{task['name']:8s} raw ds: {task['raw_ds']}")

organoid images: Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\organoid\tiff
organoid labels: Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\organoid\labels
organoid raw ds: Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\nnUNet\nnUNet_raw\Dataset001_Organoid
rosette  images: Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\tiff\completed
rosette  labels: Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\rosette\labels\completed
rosette  raw ds: Y:\Patryk_Polinski\Bel_Marta_Classification\labels_for_training\nnUNet\nnUNet_raw\Dataset002_Rosette


## Convert both tasks to nnU-Net raw format

In [42]:
for task in tasks:
    images_dir, labels_dir = task['images_dir'], task['labels_dir']
    image_glob, label_ext, raw_ds = task['image_glob'], task['label_ext'], task['raw_ds']
    dataset_name = task['dataset_name']

    img_paths = sorted(images_dir.glob(image_glob))
    missing_labels = [p.name for p in img_paths if not (labels_dir / (p.stem + label_ext)).exists()]
    print(f'{dataset_name}: {len(img_paths)} images found; {len(missing_labels)} without a matching label',
          missing_labels[:5])

    bad_spacing = check_pixel_metadata(img_paths)
    if bad_spacing:
        raise ValueError(f'{dataset_name}: {len(bad_spacing)} image(s) missing pixel-size metadata -> '
                          'remove or fix them, then rerun')

    build_dataset(img_paths, labels_dir, label_ext, raw_ds)
    n_train = len(list((raw_ds / 'labelsTr').glob('*.nii.gz')))
    generate_dataset_json(str(raw_ds), channel_names=task['channel_names'], labels=task['labels'],
                          num_training_cases=n_train, file_ending='.nii.gz', dataset_name=dataset_name)
    task['n_train'] = n_train
    print(f'{dataset_name}: dataset.json written with {n_train} training cases')

Dataset001_Organoid: 167 images found; 0 without a matching label []
all 167 images have pixel-size metadata
  case_000  r01c02f01p01-ch1sk1fk1fl1                     foreground= 0.00%  spacing_xy=(0.006449999987718184, 0.006449999987718184)
  case_001  r01c02f06p01-ch1sk1fk1fl1                     foreground= 0.17%  spacing_xy=(0.006449999987718184, 0.006449999987718184)
  case_002  r01c02f08p01-ch1sk1fk1fl1                     foreground= 0.00%  spacing_xy=(0.006449999987718184, 0.006449999987718184)
  case_003  r01c02f09p01-ch1sk1fk1fl1                     foreground= 0.00%  spacing_xy=(0.006449999987718184, 0.006449999987718184)
  case_004  r01c02f19p01-ch1sk1fk1fl1                     foreground= 1.04%  spacing_xy=(0.006449999987718184, 0.006449999987718184)
  case_005  r01c02f19p01-ch2sk1fk1fl1                     foreground= 1.40%  spacing_xy=(0.006449999987718184, 0.006449999987718184)
  case_006  r01c02f23p01-ch2sk1fk1fl1                     foreground= 0.27%  spacing_xy=(0.00

## Plan, preprocess (both tasks)

In [43]:
for task in tasks:
    run(f"nnUNetv2_plan_and_preprocess -d {task['dataset_id']} --verify_dataset_integrity")
    plans = nnunet_dirs['preprocessed'] / task['dataset_name'] / 'nnUNetPlans.json'
    if plans.exists():
        cfg = json.loads(plans.read_text())['configurations'].get(task['config'], {})
        print(f"{task['name']}: target spacing={cfg.get('spacing')}  "
              f"patch={cfg.get('patch_size')}  batch={cfg.get('batch_size')}")

> nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity
Fingerprint extraction...
Dataset001_Organoid
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

Extracting dataset fingerprint: 100%|██████████| 167/167 [00:19<00:00,  8.66it/s]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 2, 'patch_size': (np.int64(1280), np.int64(1024)), 'median_im

## Train (run one task at a time)

Set `task` to `tasks[0]` (organoid) or `tasks[1]` (rosette), then run the cells below. Repeat for the other task once this one finishes.

In [ ]:
task = tasks[0]   

preview_dir = task['images_dir'].parent / 'training_previews'
preview_every = 10                       # save one preview every N epochs
preview_dir.mkdir(parents=True, exist_ok=True)
os.environ['NNUNET_PREVIEW_DIR']   = str(preview_dir)
os.environ['NNUNET_PREVIEW_EVERY'] = str(preview_every)

# Long-running. da_workers=0 avoids the silently-dying augmentation workers on Windows.
# You can also run this exact command in a `vascumap_nnunet` terminal instead.
run(f"nnUNetv2_train {task['dataset_id']} {task['config']} {task['fold']} -tr {task['trainer']}", da_workers=0)

> nnUNetv2_train 1 2d all -tr nnUNetTrainer_250epochs_preview

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-09-22 17:42:57.270276: do_dummy_2d_data_aug: False

This is the configuration used by this training:
Configuration name: 2d
 {'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 2, 'patc

In [ ]:
from IPython.display import Image, display
run_dir = nnunet_dirs['results'] / task['dataset_name'] / f"{task['trainer']}__nnUNetPlans__{task['config']}" / f"fold_{task['fold']}"
prog = run_dir / 'progress.png'
display(Image(str(prog))) if prog.exists() else print('no progress.png yet:', run_dir)

# Latest input / ground-truth / prediction preview saved next to the data.
previews = sorted(preview_dir.glob('preview_epoch_*.png'))
display(Image(str(previews[-1]))) if previews else print('no preview PNG yet in', preview_dir)

In [ ]:
task = tasks[1]   # EDIT: switch to tasks[1] to train the rosette model instead

preview_dir = task['images_dir'].parent / 'training_previews'
preview_every = 10                       # save one preview every N epochs
preview_dir.mkdir(parents=True, exist_ok=True)
os.environ['NNUNET_PREVIEW_DIR']   = str(preview_dir)
os.environ['NNUNET_PREVIEW_EVERY'] = str(preview_every)

# Long-running. da_workers=0 avoids the silently-dying augmentation workers on Windows.
# You can also run this exact command in a `vascumap_nnunet` terminal instead.
run(f"nnUNetv2_train {task['dataset_id']} {task['config']} {task['fold']} -tr {task['trainer']}", da_workers=0)

In [ ]:

run_dir = nnunet_dirs['results'] / task['dataset_name'] / f"{task['trainer']}__nnUNetPlans__{task['config']}" / f"fold_{task['fold']}"
prog = run_dir / 'progress.png'
display(Image(str(prog))) if prog.exists() else print('no progress.png yet:', run_dir)

# Latest input / ground-truth / prediction preview saved next to the data.
previews = sorted(preview_dir.glob('preview_epoch_*.png'))
display(Image(str(previews[-1]))) if previews else print('no preview PNG yet in', preview_dir)

## Predict on a folder of new brightfield images

Point `predict_src` at a folder of new `*.tiff` brightfield frames for the currently-selected `task`. Each is staged as a `(1,Y,X)` NIfTI using its own header pixel size, predicted, and written back out as a binary foreground mask `*.tif`.

In [ ]:
# predict_src = task['images_dir']                        # EDIT: folder of new brightfield *.tiff to segment
# pred_in  = nnunet_base / 'predict_in'
# pred_out = nnunet_base / 'predict_out'
# pred_tif = nnunet_base / 'predict_masks'
# predict_tifs = sorted(predict_src.glob('*.tiff'))
# print(f'{len(predict_tifs)} images to segment')
# segment_folder(predict_tifs, pred_in, pred_out, pred_tif, task['dataset_id'], task['config'], task['fold'],
#                 task['trainer'], task['checkpoint'], task['foreground_label_value'])

## Predict on time-lapse stacks

Segments every slice of each stack in `stack_src` with the currently-selected `task`'s model, then saves a 2-channel (ch0=brightfield, ch1=foreground label) stack to `stack_out`.

In [ ]:
import shutil

stack_src = Path('C:/Users/isobe/The University of Manchester Dropbox/Isobel Taylor-Hearn/marta_labels/stack_videos')
stack_out = Path('C:/Users/isobe/The University of Manchester Dropbox/Isobel Taylor-Hearn/marta_labels/stack_outputs')
stack_in_nii = nnunet_base / 'stack_in'
stack_pred   = nnunet_base / 'stack_pred'
for d in (stack_in_nii, stack_pred):
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True, exist_ok=True)

stack_tifs = sorted(stack_src.rglob('*.tif*'))
bad_spacing = check_pixel_metadata(stack_tifs)
if bad_spacing:
    raise ValueError(f'{len(bad_spacing)} stack(s) missing pixel-size metadata -> remove or fix them, then rerun')

# Stage every slice of every stack as an independent 2D case (a single predict call for all).
counts = {}
for i, p in enumerate(stack_tifs):
    spacing_xy, counts[i] = stage_stack(p, stack_in_nii, i)
    print(f'  {p.name}  spacing_xy={spacing_xy}  slices={counts[i]}')
print(f'{len(stack_tifs)} stacks, {sum(counts.values())} slices staged; predicting...')

run(f"nnUNetv2_predict -i {stack_in_nii} -o {stack_pred} -d {task['dataset_id']} -c {task['config']} "
    f"-f {task['fold']} -tr {task['trainer']} -chk {task['checkpoint']}")

# Reassemble per stack: ZCYX with ch0=brightfield, ch1=foreground label; mirror sub-folders.
for i, p in enumerate(stack_tifs):
    out, labels = reassemble_stack(p, stack_pred, i, counts[i], task['foreground_label_value'], stack_src, stack_out)
    print(f'{p.relative_to(stack_src).as_posix()}  ({counts[i]} slices)  '
          f'foreground%={100 * float(labels.mean()):.2f}  -> {out}')
print('done ->', stack_out)